**<h1>Classification Model - V3**
#### Classification Model V3 =  
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 06/24/2025</i>
* AHJ Data: City and County level data sourced from Government Compensation California (GCC)
* Used to create filtered outputs from test/train data
* Started development on 06/24/2025
* Finished Iteration on -06/24/2025-

In [2]:
# Run pip updates and installs here
%pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [3]:
# Import libraries
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords
nltk.download('stopwords')


C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rford\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
# Keywords to KEEP
keep_dept_keywords = [
    "Planning", "Community Development", "Building Department", "Code Compliance",
    "Public Works", "Streets", "Engineering", "Construction", "Zoning"
]

keep_position_keywords = [
    "Construction", "Building", "HVAC", "Plans Inspector", "Zoning Investigator",
    "Architect", "Permit Technician", "Safety Inspector", "Planner", "Code Enforcer",
    "Plan Check Coordinator", "Licensing Specialist", "Community Development Technician"
]

# Keywords to FILTER OUT
filter_dept_keywords = [
    "Administration", "Communications", "Animal", "Finance", "Community Services",
    "Municipal Power", "Base Reuse", "City Attorney", "City Clerk", "Council", "Fire",
    "Human Resources", "Library", "IT", "Police", "Recreation", "Government Services",
    "Facilities", "Water", "Power", "Airport", "Health", "Zero Waste", "Sewer",
    "Equipment", "Transfer Station", "Janitor", "Landscaping", "Utilities", "Business",
    "Transit", "Nutrition", "Aging", "Adult", "Youth", "Homeless", "Athletics", "Law",
    "Attorney", "Management"
]

filter_position_keywords = [
    "Fitness instructor", "Recreational Leader", "Librarian", "Admin", "Clerk",
    "Secretary", "Parking Officer", "Police", "Firefighter", "Treasurer", "Employment Worker",
    "Lifeguard", "Animal", "Recycling", "Sewer", "Finance", "HR", "IT", "Parks & Rec",
    "Health", "EMT", "Custodian", "Equipment Mechanic", "Aquatics", "Traffic",
    "Community Service", "Veterinary", "Senior Citizen", "Pool", "Utilities", "Audio",
    "Communications", "Mayor", "Attendant", "Vocational Worker", "Messenger Clerk",
    "Customer Service", "Truck Operator", "Sanitation", "Painter", "Eltl Engr Assoc",
    "Laborer", "Program Assistant"
]


In [5]:
# Helper function to match keywords
def keyword_match(text, keywords):
    if pd.isna(text):
        return False
    return any(re.search(rf'\b{re.escape(k)}\b', str(text), re.IGNORECASE) for k in keywords)

# Label rows for training
def label_row(row):
    keep_dept = keyword_match(row['DepartmentOrSubdivision'], keep_dept_keywords)
    keep_pos = keyword_match(row['Position'], keep_position_keywords)
    filter_dept = keyword_match(row['DepartmentOrSubdivision'], filter_dept_keywords)
    filter_pos = keyword_match(row['Position'], filter_position_keywords)
    return int((keep_dept or keep_pos) and not (filter_dept or filter_pos))


In [6]:
# Load training data
train_df = pd.read_excel("city_training_data.xlsx")

# Apply labeling
train_df["label"] = train_df.apply(label_row, axis=1)

# Combine text for model training
train_df["combined_text"] = train_df["DepartmentOrSubdivision"].fillna('') + " " + train_df["Position"].fillna('')

# Show label distribution
train_df["label"].value_counts()


label
0    91664
1    11646
Name: count, dtype: int64

In [7]:
# Feature and Target Preparation

X = train_df["combined_text"]
y = train_df["label"]


In [8]:
# Generate Sentence Embeddings

embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train = embedder.encode(train_df["combined_text"].tolist(), show_progress_bar=True)
y_train = train_df["label"]


C:\Users\Rford\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rford\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falli

In [9]:
# Train the Classifier

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

print("✅ Model trained successfully.")


✅ Model trained successfully.


In [10]:
# Load test data
test_df = pd.read_excel("city_test_data.xlsx")

# Combine text columns
test_df["combined_text"] = test_df["DepartmentOrSubdivision"].fillna('') + " " + test_df["Position"].fillna('')

# Predict on Test Data
X_test = embedder.encode(test_df["combined_text"].tolist(), show_progress_bar=True)
test_df["predicted_label"] = clf.predict(X_test)

# Filter the results
filtered_test_df = test_df[test_df["predicted_label"] == 1]

# Display summary
print(f"Filtered rows: {len(filtered_test_df)} / {len(test_df)}")
filtered_test_df.head()


Batches: 100%|██████████| 7533/7533 [08:45<00:00, 14.33it/s]


Filtered rows: 25286 / 241056


,Year,EmployerType,EmployerName,DepartmentOrSubdivision,Position,ElectedOfficial,Judicial,OtherPositions,MinPositionSalary,MaxPositionSalary,...,PensionFormula,EmployerURL,EmployerPopulation,LastUpdatedDate,EmployerCounty,SpecialDistrictActivities,IncludesUnfundedLiability,SpecialDistrictType,combined_text,predicted_label
20,2023,City,Adelanto,Streets,Maint Worker I,False,False,NaN,44363.0,49931.0,...,2%@62,https://www.ci.adelanto.ca.us/198/Human-Resources,36131,2024-06-25,San Bernardino,NaN,False,NaN,Streets Maint Worker I,1
38,2023,City,Agoura Hills,Community Development,Associate Planner,False,False,NaN,98041.0,119454.0,...,2%@55,https://www.agourahillscity.org/department/hum...,19841,2024-06-25,Los Angeles,NaN,False,NaN,Community Development Associate Planner,1
39,2023,City,Agoura Hills,Community Development,Associate Planner,False,False,NaN,98041.0,119454.0,...,2%@62,https://www.agourahillscity.org/department/hum...,19841,2024-06-25,Los Angeles,NaN,False,NaN,Community Development Associate Planner,1
40,2023,City,Agoura Hills,Community Development,Building Official,False,False,NaN,141993.0,173005.0,...,2%@62,https://www.agourahillscity.org/department/hum...,19841,2024-06-25,Los Angeles,NaN,False,NaN,Community Development Building Official,1
41,2023,City,Agoura Hills,Community Development,Community Development Director,False,False,NaN,173005.0,210789.0,...,2%@62,https://www.agourahillscity.org/department/hum...,19841,2024-06-25,Los Angeles,NaN,False,NaN,Community Development Community Development Di...,1


In [11]:
# Save the filtered test data
filtered_test_df.to_excel("city_filtered_test_data_v3.xlsx", index=False)
print("✅ Filtered test data saved as 'city_filtered_test_data_v3.xlsx'")


✅ Filtered test data saved as 'city_filtered_test_data_v3.xlsx'
